# 11.11 平均高分掩蓋了哪些弱項？

# 第 11 章：對齊、問答與語言能力保留

前置：第10章介面與第7章SFT。能力實驗需先有「文字屬性→正確答案」基模，以及可分類的視覺encoder；用scripts/prepare_data.py和pretrain_encoders.py準備。下面隨機模型只驗證資料／梯度，所有品質留給讀者訓練後量測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：轉接頭尺寸合了，還要學資料的意思**

先確認文字基模會用文字屬性回答，再學圖片接頭。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/alignment.svg")))

**先想一想：**常見組90題全對，稀少組10題全錯，總分掩蓋什麼？

平均分會被常見類別主導。按顏色、形狀與稀少組合列正確數/總數，避免把總體高分當成每個子任務都可靠。

**把直覺寫成數學：**micro accuracy按例子加權；macro accuracy按組平均，兩者用途不同。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
groups = {"常見": {"correct": 90, "count": 90}, "稀少": {"correct": 0, "count": 10}}
micro = sum(g["correct"] for g in groups.values()) / sum(g["count"] for g in groups.values())
macro = sum(g["correct"] / g["count"] for g in groups.values()) / len(groups)
print("micro", micro, "macro", macro, groups)

**如何讀結果：**數字是明確構造的反例，不是報告trained模型結果。

**只改一件事：**只改變稀少組樣本數，查看macro爲什麼不隨頻率變。
